In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report 
from sklearn.naive_bayes import MultinomialNB
from sklearn import svm
import regex as re
import scipy
from datetime import datetime
import torch
from torch import nn
import tensorflow_hub as hub
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from datasets import Dataset
import os
import random
import numpy as np
import evaluate
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# Data Prep

In [2]:
def get_ngrams(data, ngrams):
    vectorizer = TfidfVectorizer(ngram_range=(1,ngrams))
    features = vectorizer.fit_transform(data)
    return features

def get_splits(x, y, random_state=12345):
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=random_state)
    x_val, x_test, y_val, y_test = train_test_split(x_test, y_test, test_size=0.5, random_state=random_state)
    return x_train, x_val, x_test, y_train, y_val, y_test 

def calc_metrics(y_true, y_pred):
    f1_macro = f1_score(y_true, y_pred, average='macro')
    accuracy = accuracy_score(y_true, y_pred)
    print(f"f1 macro: {f1_macro}\n accuracy: {accuracy}")
    

data_path = "data/raw/rusentiment_preselected_posts.csv"
data1 = pd.read_csv(data_path )
data_path2 = "data/raw/rusentiment_random_posts.csv"
data2 = pd.read_csv(data_path2)
data_all = pd.concat((data1, data2))
target_classes = ["neutral", "positive", "negative"]
data = data_all[data_all["label"].isin(target_classes)]
random_state=12345

# LR + bigrams

In [4]:
# features prep
features = get_ngrams(data["text"], 2)
x_train, x_val, x_test, y_train, y_val, y_test = get_splits(features, data["label"], random_state=random_state)

# train
model = LogisticRegression(max_iter = 100, random_state=random_state, solver="saga", l1_ratio=0.5)
model.fit(x_train, y_train)


# eval
y_pred = model.predict(x_train)
print("train eval")
calc_metrics(y_train, y_pred)

y_pred = model.predict(x_test)
print("test eval")
calc_metrics(y_test, y_pred)

train eval
f1 macro: 0.5471915718449224
 accuracy: 0.6762209957866002
test eval
f1 macro: 0.5034852764938906
 accuracy: 0.6449928808732796


# SVM + trigrams

In [5]:
# features prep
features = get_ngrams(data["text"], 3)
x_train, x_val, x_test, y_train, y_val, y_test = get_splits(features, data["label"], random_state=random_state)

# train
model = svm.LinearSVC(random_state=random_state, C=0.3, tol=1e-3, penalty="l1")
model.fit(x_train, y_train)


# eval
y_pred = model.predict(x_train)
print("train eval")
calc_metrics(y_train, y_pred)

y_pred = model.predict(x_test)
print("test eval")
calc_metrics(y_test, y_pred)

train eval
f1 macro: 0.5395810317432058
 accuracy: 0.6705240045101181
test eval
f1 macro: 0.5066631924488689
 accuracy: 0.6459420977693403


# USE + LR

In [6]:
embeding_model = hub.load("https://www.kaggle.com/models/google/universal-sentence-encoder/tensorFlow2/universal-sentence-encoder/2?tfhub-redirect=true")
features = embeding_model(data["text"]).numpy()

In [7]:

x_train, x_val, x_test, y_train, y_val, y_test = get_splits(features, data["label"], random_state=random_state)

# train
model = LogisticRegression(max_iter = 300, random_state=random_state, solver="lbfgs")
model.fit(x_train, y_train)


# eval
y_pred = model.predict(x_train)
print("train eval")
calc_metrics(y_train, y_pred)

y_pred = model.predict(x_test)
print("test eval")
calc_metrics(y_test, y_pred)

train eval
f1 macro: 0.358610914439698
 accuracy: 0.5616283900065278
test eval
f1 macro: 0.34916415026649567
 accuracy: 0.546748932130992


# NaiveBayes

In [8]:
# features prep
features = get_ngrams(data["text"], 1)
x_train, x_val, x_test, y_train, y_val, y_test = get_splits(features, data["label"], random_state=random_state)

# train
model = MultinomialNB()
model.fit(x_train, y_train)


# eval
y_pred = model.predict(x_train)
print("train eval")
calc_metrics(y_train, y_pred)

y_pred = model.predict(x_test)
print("test eval")
calc_metrics(y_test, y_pred)

train eval
f1 macro: 0.5628959850482489
 accuracy: 0.718889086701086
test eval
f1 macro: 0.4017785064353074
 accuracy: 0.608922638822971


# Authorship method

# Finetuning XLM-R

In [3]:
train_df = data.sample(frac=0.8, random_state=random_state)
test_df = data.drop(train_df.index)
val_df = test_df.sample(frac=0.5, random_state=random_state)
test_df = test_df.drop(val_df.index)

In [17]:


MODEL_NAME = "FacebookAI/xlm-roberta-base"
OUTPUT_DIR = "./xlmr_rusentiment_quick_2e-5_p1_normalized"
MAX_LENGTH = 256
SEED = random_state
USE_WANDB = True
RUN_NAME = "xlmr-rusentiment-quick_2e-5_p1_normalized"
TRAIN_BS = 16
EVAL_BS = 32
EPOCHS = 10
LR = 1e-5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


if pd.api.types.is_integer_dtype(train_df["label"]) and pd.api.types.is_integer_dtype(test_df["label"]):
    label_values = sorted(set(train_df["label"].unique()).union(set(test_df["label"].unique())))
    label2id = {int(v): int(v) for v in label_values}
    id2label = {int(v): str(v) for v in label_values}
    train_df["label"] = train_df["label"].astype(int)
    test_df["label"] = test_df["label"].astype(int)
else:
    all_labels = sorted(set(train_df["label"].astype(str).unique()).union(set(test_df["label"].astype(str).unique())))
    label2id = {label: idx for idx, label in enumerate(all_labels)}
    id2label = {idx: label for label, idx in label2id.items()}
    train_df["label"] = train_df["label"].astype(str).map(label2id).astype(int)
    test_df["label"] = test_df["label"].astype(str).map(label2id).astype(int)

num_labels = len(id2label)
print("label2id:", label2id)
print("train distribution:\n", train_df["label"].value_counts().sort_index())
print("test distribution:\n", test_df["label"].value_counts().sort_index())

classes = np.array(sorted(train_df["label"].unique()))
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
class_weights = torch.nn.functional.normalize(class_weights, p=1, dim=0)
print("class weights:", class_weights.tolist())

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"])

train_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
test_dataset = test_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id={str(k): v for k, v in label2id.items()} if not all(isinstance(k, str) for k in label2id) else label2id,
)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
        "f1_weighted": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=10,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    logging_strategy="steps",
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    bf16=False,
    dataloader_num_workers=0,
    seed=SEED,
    remove_unused_columns=False,
)


trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=30)],
)

train_result = trainer.train()
print(train_result)

metrics = trainer.evaluate()
print("Eval metrics:", metrics)

pred_output = trainer.predict(test_dataset)
y_true = pred_output.label_ids
y_pred = pred_output.predictions.argmax(axis=-1)

print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    digits=4,
    target_names=[id2label[i] for i in sorted(id2label)]
))

trainer.save_model(os.path.join(OUTPUT_DIR, "best_model_v2"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "best_model_v2"))

label2id: {0: 0, 1: 1, 2: 2}
train distribution:
 label
0    2893
1    9066
2    4892
Name: count, dtype: int64
test distribution:
 label
0    204
1    748
2    403
Name: count, dtype: int64
class weights: [0.5234292149543762, 0.1670285314321518, 0.3095422387123108]


Map:   0%|          | 0/16851 [00:00<?, ? examples/s]

Map:   0%|          | 0/1355 [00:00<?, ? examples/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
100,1.106700,1.096852,0.552030,0.237122,0.392694
200,1.113900,1.093395,0.552030,0.237122,0.392694
300,1.090700,1.073316,0.616974,0.427818,0.561404
400,1.057800,0.963629,0.643542,0.587188,0.650413
500,0.959600,0.828898,0.707011,0.659431,0.708985
600,0.830200,0.788066,0.701845,0.662735,0.703006
700,0.820100,0.677389,0.726199,0.697673,0.729076
800,0.725900,0.684396,0.709963,0.686806,0.715990
900,0.715800,0.678262,0.738745,0.709103,0.742862
1000,0.706700,0.645585,0.695941,0.681377,0.702696


TrainOutput(global_step=6000, training_loss=0.5054166523615519, metrics={'train_runtime': 2091.2842, 'train_samples_per_second': 80.577, 'train_steps_per_second': 5.04, 'total_flos': 5229450094632372.0, 'train_loss': 0.5054166523615519, 'epoch': 5.692599620493358})


Eval metrics: {'eval_loss': 0.5348255634307861, 'eval_accuracy': 0.8051660516605166, 'eval_f1_macro': 0.781342968159902, 'eval_f1_weighted': 0.8083658083592041, 'eval_runtime': 1.4306, 'eval_samples_per_second': 947.126, 'eval_steps_per_second': 30.056, 'epoch': 5.692599620493358}

Confusion matrix:
[[160  29  15]
 [ 78 585  85]
 [ 20  37 346]]

Classification report:
              precision    recall  f1-score   support

           0     0.6202    0.7843    0.6926       204
           1     0.8986    0.7821    0.8363       748
           2     0.7758    0.8586    0.8151       403

    accuracy                         0.8052      1355
   macro avg     0.7649    0.8083    0.7813      1355
weighted avg     0.8202    0.8052    0.8084      1355



eval/accuracy,▁▃▃▅▅▅▆▅▄▅▇▅▇▆█▇▆▇███▇█▇▇██▇███▇▇████▇██
eval/f1_macro,▁▁▃▆▆▇▇▇▇▆▇▇▇▇▇█████████████████████████
eval/f1_weighted,▁▁▄▅▆▇▆▇▆▆▇▆▇▇█▇████████████████████████
eval/loss,██▆▅▄▃▂▂▂▂▁▂▂▁▁▂▂▂▁▁▁▂▂▃▂▂▃▂▂▃▃▃▃▃▄▄▄▆▆▅
eval/runtime,▂▁▂▂▅▃▄▃▄▂▅▂▃▃▃▂▁▂▃▁▃▂▂▂▂▂▂▁▂▃▂▂▂▃▄▂▂▁▁█
eval/samples_per_second,▆█▇▄▆▆▆▅▇█▆▅▇▅▄▆▆▅▆▆▇▇▅▅▆▆▇▇▇▇▆▇▇▅▆▇▇█▇▁
eval/steps_per_second,▆█▆▇▆▆▅▆▆▅█▅▇▄▆▅▅▇▆█▅▆▇▆▆▇▇█▇▆▆▇▇▅▅▇▇█▇▁
test/accuracy,▁
test/f1_macro,▁
test/f1_weighted,▁
+9,...


# Finetuning XLM-V

In [5]:
import os
import random
import numpy as np
import pandas as pd
import torch
import evaluate
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)


MODEL_NAME = "facebook/xlm-v-base"
OUTPUT_DIR = "./xlmv_cosine_ws"
MAX_LENGTH = 256
SEED = random_state
RUN_NAME = "xlmv-same_cosine_LabalSmoothing, lr1e-5 warmpup 0.05 cosine cycle"
TRAIN_BS = 8
EVAL_BS = 32
EPOCHS = 4
LR = 1e-5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


if pd.api.types.is_integer_dtype(train_df["label"]) and pd.api.types.is_integer_dtype(test_df["label"]):
    label_values = sorted(set(train_df["label"].unique()).union(set(test_df["label"].unique())))
    label2id = {int(v): int(v) for v in label_values}
    id2label = {int(v): str(v) for v in label_values}
    train_df["label"] = train_df["label"].astype(int)
    test_df["label"] = test_df["label"].astype(int)
else:
    all_labels = sorted(set(train_df["label"].astype(str).unique()).union(set(test_df["label"].astype(str).unique())))
    label2id = {label: idx for idx, label in enumerate(all_labels)}
    id2label = {idx: label for label, idx in label2id.items()}
    train_df["label"] = train_df["label"].astype(str).map(label2id).astype(int)
    test_df["label"] = test_df["label"].astype(str).map(label2id).astype(int)

num_labels = len(id2label)
print("label2id:", label2id)
print("train distribution:\n", train_df["label"].value_counts().sort_index())
print("test distribution:\n", test_df["label"].value_counts().sort_index())

classes = np.array(sorted(train_df["label"].unique()))
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
class_weights = torch.nn.functional.normalize(class_weights, p=1, dim=0)
print("class weights:", class_weights.tolist())

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"])

train_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
test_dataset = test_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id={str(k): v for k, v in label2id.items()} if not all(isinstance(k, str) for k in label2id) else label2id,
)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
        "f1_weighted": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    learning_rate=LR,
    weight_decay=0.01,
    label_smoothing_factor=0.1,
    warmup_steps=500,
    lr_scheduler_kwargs={"num_cycles": 3},
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=1000,
    logging_strategy="steps",
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    bf16=False,
    dataloader_num_workers=0,
    seed=SEED,
    remove_unused_columns=False,
)


trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,  
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=30)],
)

train_result = trainer.train()
print(train_result)

metrics = trainer.evaluate()
print("Eval metrics:", metrics)

pred_output = trainer.predict(test_dataset)
y_true = pred_output.label_ids
y_pred = pred_output.predictions.argmax(axis=-1)

print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    digits=4,
    target_names=[id2label[i] for i in sorted(id2label)]
))

trainer.save_model(os.path.join(OUTPUT_DIR, "best_model_v2"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "best_model_v2"))


label2id: {'negative': 0, 'neutral': 1, 'positive': 2}
train distribution:
 label
0    2893
1    9066
2    4892
Name: count, dtype: int64
test distribution:
 label
0    204
1    748
2    403
Name: count, dtype: int64
class weights: [0.5234292149543762, 0.1670285314321518, 0.3095422387123108]


Map:   0%|          | 0/16851 [00:00<?, ? examples/s]

Map:   0%|          | 0/1355 [00:00<?, ? examples/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at facebook/xlm-v-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Darkmotion\_netrc.
wandb: Currently logged in as: alephnull to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
100,1.102300,1.105166,0.150554,0.087235,0.039401
200,1.104700,1.103367,0.150554,0.087235,0.039401
300,1.103500,1.094737,0.526199,0.304352,0.445182
400,1.098000,1.097192,0.265683,0.210069,0.233823
500,1.087200,1.064536,0.357934,0.231274,0.259866
600,1.079000,1.083387,0.380074,0.254410,0.294101
700,1.087100,1.088373,0.304059,0.161926,0.150493
800,1.077600,1.040739,0.538007,0.387870,0.494113
900,1.057200,0.949667,0.609594,0.468851,0.574023
1000,0.996500,0.908625,0.667159,0.572211,0.651731


TrainOutput(global_step=8428, training_loss=0.671174652948728, metrics={'train_runtime': 2393.5363, 'train_samples_per_second': 28.161, 'train_steps_per_second': 3.521, 'total_flos': 2336373662669298.0, 'train_loss': 0.671174652948728, 'epoch': 4.0})


Eval metrics: {'eval_loss': 0.6707390546798706, 'eval_accuracy': 0.7911439114391144, 'eval_f1_macro': 0.7551995525318507, 'eval_f1_weighted': 0.7926378888416572, 'eval_runtime': 1.2388, 'eval_samples_per_second': 1093.835, 'eval_steps_per_second': 34.712, 'epoch': 4.0}

Confusion matrix:
[[136  45  23]
 [ 68 612  68]
 [ 18  61 324]]

Classification report:
              precision    recall  f1-score   support

    negative     0.6126    0.6667    0.6385       204
     neutral     0.8524    0.8182    0.8349       748
    positive     0.7807    0.8040    0.7922       403

    accuracy                         0.7911      1355
   macro avg     0.7486    0.7629    0.7552      1355
weighted avg     0.7950    0.7911    0.7926      1355



eval/accuracy,▁▁▂▃▃▆▇▇▇▇▇▇▇▇▇▇▇█▇▇███████████▇████████
eval/f1_macro,▁▃▅▆▇▇▇▇▇▇▇▇▇▇▇██████████▇█▇████████████
eval/f1_weighted,▁▁▃▃▂▇▇▇▇▇▇█▇▇██████████████▇▇███████▇██
eval/loss,██▇▅▄▂▂▂▂▂▂▂▂▂▃▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▁▂▃▂▃▂
eval/runtime,▂▅▁▅▂▂▃▂▂▂▅▄▂▂▂▆▃▂▂▄█▁▃▂▃▆▄▂▂▅▁▂▁▂▁▂▂▁▂█
eval/samples_per_second,▇█▃▇▂▆▇▇▇▅▅▇▇▅▇▃▅▇▆▃▇▇▄▄▂█▆▄▄▇▇▄▇████▇▇▁
eval/steps_per_second,▆▃▆▅▇▁▄▄▆▄▇▂▆▇▇▂▅▇▄▁█▅▆▃▅▂▁▄▃▃▆▇█▇█▇▇▇▇▆
test/accuracy,▁
test/f1_macro,▁
test/f1_weighted,▁
+9,...


In [95]:
def predict_labels_xlmr(df, model, tokenizer, text_col="text", batch_size=32, max_length=192, device=None):

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)
    model.eval()

    texts = df[text_col].astype(str).tolist()
    pred_ids = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            batch_pred_ids = torch.argmax(outputs.logits, dim=-1).cpu().tolist()

        pred_ids.extend(batch_pred_ids)

    if hasattr(model.config, "id2label") and model.config.id2label:
        return [model.config.id2label[int(i)] for i in pred_ids]

    return pred_ids

In [198]:
def predict_proba_xlmr(df, model, tokenizer, text_col="text", batch_size=32, max_length=192, device=None):
    import torch

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)
    model.eval()

    texts = df[text_col].astype(str).tolist()
    all_proba = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = outputs.logits.cpu().numpy()

        all_proba.append(probs)

    import numpy as np
    return np.vstack(all_proba)

In [216]:
y_pred_probs_train = predict_proba_xlmr(train_df, model, tokenizer, text_col="text", batch_size=32, max_length=256, device=None)